In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import requests
import time
import logging
from typing import Dict, Any, Optional

logging.basicConfig(level=logging.INFO)

class RateLimiter:
    def __init__(self, calls_per_minute: int):
        self.calls_per_minute = calls_per_minute
        self.call_times = []

    def wait(self):
        now = time.time()
        self.call_times = [t for t in self.call_times if now - t < 60]
        if len(self.call_times) >= self.calls_per_minute:
            sleep_time = 60 - (now - self.call_times[0])
            logging.info(f"Rate limit reached. Sleeping for {sleep_time:.2f} seconds.")
            time.sleep(sleep_time)
        self.call_times.append(time.time())

class StockAPIClient:
    def fetch(self, symbol: str) -> Optional[Dict[str, Any]]:
        raise NotImplementedError

class AlphaVantageClient(StockAPIClient):
    BASE_URL = "https://www.alphavantage.co/query"

    def __init__(self, api_key: str, rate_limiter: RateLimiter):
        self.api_key = api_key
        self.rate_limiter = rate_limiter

    def fetch(self, symbol: str) -> Optional[Dict[str, Any]]:
        self.rate_limiter.wait()
        params = {
            "function": "TIME_SERIES_INTRADAY",
            "symbol": symbol,
            "interval": "1min",
            "apikey": self.api_key
        }
        response = requests.get(self.BASE_URL, params=params)
        if response.status_code == 200:
            return response.json()
        logging.error(f"AlphaVantage error: {response.status_code}")
        return None

class YahooFinanceClient(StockAPIClient):
    BASE_URL = "https://query1.finance.yahoo.com/v8/finance/chart/"

    def fetch(self, symbol: str) -> Optional[Dict[str, Any]]:
        url = f"{self.BASE_URL}{symbol}"
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        logging.error(f"YahooFinance error: {response.status_code}")
        return None

# Example usage:
# av_client = AlphaVantageClient(api_key="YOUR_API_KEY", rate_limiter=RateLimiter(5))
# data = av_client.fetch("AAPL")
# yf_client = YahooFinanceClient()
# data = yf_client.fetch("AAPL")

In [3]:
import pandas as pd
import numpy as np

def calculate_sma(series: pd.Series, window: int) -> pd.Series:
    """Simple Moving Average"""
    return series.rolling(window=window).mean()

def calculate_ema(series: pd.Series, window: int) -> pd.Series:
    """Exponential Moving Average"""
    return series.ewm(span=window, adjust=False).mean()

def calculate_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    """Relative Strength Index"""
    delta = series.diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)
    avg_gain = pd.Series(gain).rolling(window=window).mean()
    avg_loss = pd.Series(loss).rolling(window=window).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return pd.Series(rsi, index=series.index)

# Example usage:
# Suppose 'data' is a dictionary from AlphaVantageClient or YahooFinanceClient containing time series data.
# For demonstration, let's create a sample DataFrame:

sample_data = {
    'timestamp': pd.date_range(start='2024-06-01', periods=30, freq='D'),
    'close': np.random.uniform(100, 200, 30)
}
df = pd.DataFrame(sample_data).set_index('timestamp')

# Calculate indicators
df['SMA_5'] = calculate_sma(df['close'], window=5)
df['EMA_5'] = calculate_ema(df['close'], window=5)
df['RSI_14'] = calculate_rsi(df['close'], window=14)

df.tail()

,close,SMA_5,EMA_5,RSI_14
timestamp,,,,
2024-06-26,168.662765,140.611382,145.767492,NaN
2024-06-27,134.888528,136.781946,142.141171,NaN
2024-06-28,114.912445,138.792718,133.064929,NaN
2024-06-29,119.393547,134.875143,128.507801,NaN
2024-06-30,173.520177,142.275492,143.511927,NaN


In [4]:
# Automate the scanning process using APScheduler and write unit/integration tests using pytest

# 1. Install required packages if not already installed
# (APScheduler and pytest)
from metagpt.tools.libs.terminal import Terminal
import sys

term = Terminal()
await term.run_command(f"{sys.executable} -m pip install apscheduler pytest")

# 2. Scheduler setup
from apscheduler.schedulers.background import BackgroundScheduler
import time

def scan_and_analyze(symbol="AAPL"):
    # Example: fetch data and calculate indicators
    # Here, we use the sample DataFrame from previous code for demonstration
    print(f"Scanning {symbol} at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    # In real use, fetch data using API client, then calculate indicators
    # For now, just print the last row of df
    print(df.tail(1))

scheduler = BackgroundScheduler()
# Schedule scan_and_analyze to run every day at 15:30 (market close, for example)
scheduler.add_job(scan_and_analyze, 'cron', hour=15, minute=30, args=["AAPL"])
scheduler.start()

print("Scheduler started. (In a real script, keep the process alive.)")

# 3. Example pytest unit tests (to be placed in test_stock_analysis.py)
test_code = """
import pytest
import pandas as pd
import numpy as np

from your_module import calculate_sma, calculate_ema, calculate_rsi

def test_calculate_sma():
    s = pd.Series([1, 2, 3, 4, 5])
    result = calculate_sma(s, window=2)
    assert np.allclose(result.dropna(), [1.5, 2.5, 3.5, 4.5])

def test_calculate_ema():
    s = pd.Series([1, 2, 3, 4, 5])
    result = calculate_ema(s, window=2)
    assert np.isclose(result.iloc[-1], 4.666666666666667)

def test_calculate_rsi():
    s = pd.Series([1, 2, 3, 4, 5, 4, 3, 2, 1, 2, 3, 4, 5, 6, 7])
    rsi = calculate_rsi(s, window=3)
    assert rsi.notnull().sum() > 0
"""

# Write the test code to a file for pytest
from metagpt.tools.libs.editor import Editor
editor = Editor()
editor.write("test_stock_analysis.py", test_code)

print("Example unit tests written to test_stock_analysis.py. Run with: pytest test_stock_analysis.py")

Scheduler started. (In a real script, keep the process alive.)
Example unit tests written to test_stock_analysis.py. Run with: pytest test_stock_analysis.py


2026-01-12 19:08:30.577 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT
2026-01-12 19:08:30.579 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT
2026-01-12 19:08:32.387 | INFO     | metagpt.tools.libs.terminal:_check_state:55 - The terminal is at:
INFO:apscheduler.scheduler:Adding job tentatively -- it will be properly scheduled when the scheduler starts
INFO:apscheduler.scheduler:Added job "scan_and_analyze" to job store "default"
INFO:apscheduler.scheduler:Scheduler started
